# 02 — Feature Engineering

**Competition:** A Cloned Airbnb Booking Prediction Competition — K353  
**Course:** COMP 468 — Abdullah Gül University

## Goal
Transform raw multi-table data (`property_info`, `listing_2016Q1`, `listing_2016Q2`) into one **flat feature matrix per `PropertyID`**, ready for modelling.

We engineer features from three angles:
1. **Listing behaviour** (Q1, Q2, and combined H1) — availability, bookings, status transitions, monthly trends, weekend patterns, recency.
2. **Pricing dynamics** — central tendency, volatility, weekend premium, percentile features.
3. **Property metadata** — type/policy flags, price-per-guest, property age, host activity, location.

Outputs:
- `outputs/train_features.parquet`
- `outputs/test_features.parquet`

These will be consumed by the modelling notebooks.

---

### Outline
1. Setup & data load
2. Listing-table feature factory
   - 2.1 Per-quarter aggregates
   - 2.2 Weekend / weekday split
   - 2.3 Monthly aggregates
   - 2.4 Status-transition counts
   - 2.5 Recency window (last 30 days of Q2)
3. Property-metadata feature factory
4. Assemble final train / test feature matrices
5. Sanity checks
6. Persist to disk

## 1. Setup & Data Load

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

RANDOM_STATE = 42

candidate_paths = [
    Path('../data'),
    Path('/kaggle/input/a-cloned-airbnb-booking-prediction-competition-k-353'),
    Path('/content/data'),
]
DATA_DIR = next((p for p in candidate_paths if p.exists()), Path('../data'))
OUT_DIR = Path('../outputs'); OUT_DIR.mkdir(exist_ok=True, parents=True)
print('DATA_DIR =', DATA_DIR.resolve())

DATA_DIR = /Users/bashkal/Desktop/Comp468-ML_in_Python/ali/ML-Final/data


In [2]:
property_info = pd.read_csv(DATA_DIR / 'property_info.csv',
                            parse_dates=['CreatedDate'], low_memory=False)
train_y  = pd.read_csv(DATA_DIR / 'reserve_2016Q3_train.csv')
test_ids = pd.read_csv(DATA_DIR / 'PropertyID_test.csv')

listing_dtype = {
    'PropertyID': 'int32',
    'Status': 'category',
    'Price': 'float32',
    'ReservationID': 'float32',
}
listing_q1 = pd.read_csv(DATA_DIR / 'listing_2016Q1.csv', dtype=listing_dtype,
                         parse_dates=['Date', 'BookedDate'])
listing_q2 = pd.read_csv(DATA_DIR / 'listing_2016Q2.csv', dtype=listing_dtype,
                         parse_dates=['Date', 'BookedDate'])

print('Shapes:', property_info.shape, train_y.shape, test_ids.shape,
      listing_q1.shape, listing_q2.shape)

Shapes: (48690, 33) (24372, 2) (24318, 1) (4185022, 6) (4404861, 6)


## 2. Listing-Table Feature Factory

We build a series of helper functions that take a long-format listing DataFrame and return a per-`PropertyID` feature DataFrame. We then run each helper on Q1, Q2, and on a combined H1 frame, prefixing column names accordingly.

### 2.1 Base per-quarter aggregates

In [3]:
def base_aggregates(df, prefix):
    df = df.copy()
    df['is_A'] = (df['Status'] == 'A').astype('int8')
    df['is_B'] = (df['Status'] == 'B').astype('int8')
    df['is_R'] = (df['Status'] == 'R').astype('int8')

    agg = df.groupby('PropertyID').agg(
        days_total=('Date', 'count'),
        days_A=('is_A', 'sum'),
        days_B=('is_B', 'sum'),
        days_R=('is_R', 'sum'),
        unique_reservations=('ReservationID', 'nunique'),
        price_mean=('Price', 'mean'),
        price_median=('Price', 'median'),
        price_std=('Price', 'std'),
        price_nunique=('Price', 'nunique'),
        price_min=('Price', 'min'),
        price_max=('Price', 'max'),
        price_q25=('Price', lambda s: s.quantile(0.25)),
        price_q75=('Price', lambda s: s.quantile(0.75)),
    ).reset_index()

    agg['booking_rate']   = agg['days_B'] / agg['days_total']
    agg['available_rate'] = agg['days_A'] / agg['days_total']
    agg['reserved_rate']  = agg['days_R'] / agg['days_total']
    agg['price_range']    = agg['price_max'] - agg['price_min']
    agg['price_cv']       = agg['price_std'] / agg['price_mean'].replace(0, np.nan)
    agg['price_iqr']      = agg['price_q75'] - agg['price_q25']

    agg.columns = ['PropertyID'] + [f'{prefix}_{c}' for c in agg.columns[1:]]
    return agg

base_q1 = base_aggregates(listing_q1, 'q1')
base_q2 = base_aggregates(listing_q2, 'q2')
print('base_q1:', base_q1.shape, '| base_q2:', base_q2.shape)
base_q1.head()

base_q1: (48690, 20) | base_q2: (48690, 20)


,PropertyID,q1_days_total,q1_days_A,q1_days_B,q1_days_R,q1_unique_reservations,q1_price_mean,q1_price_median,q1_price_std,q1_price_nunique,q1_price_min,q1_price_max,q1_price_q25,q1_price_q75,q1_booking_rate,q1_available_rate,q1_reserved_rate,q1_price_range,q1_price_cv,q1_price_iqr
0,105,91,91,0,0,0,649.000000,549.0,158.989867,2,549.0,899.0,549.0,899.0,0.000000,1.000000,0.000000,350.0,0.244977,350.0
1,795,60,60,0,0,0,300.000000,300.0,0.000000,1,300.0,300.0,300.0,300.0,0.000000,1.000000,0.000000,0.0,0.000000,0.0
2,2515,91,34,22,35,5,67.681320,60.0,18.092896,29,46.0,145.0,59.0,76.0,0.241758,0.373626,0.384615,99.0,0.267325,17.0
3,2534,91,0,0,91,5,158.131866,165.0,18.403628,4,120.0,235.0,145.0,165.0,0.000000,0.000000,1.000000,115.0,0.116382,20.0
4,2539,91,90,0,1,1,55.263737,49.0,15.609709,3,39.0,79.0,49.0,79.0,0.000000,0.989011,0.010989,40.0,0.282458,30.0


### 2.2 Weekend / weekday split
Weekend booking patterns differ markedly from weekday ones (Fri-Sat-Sun premium). Capture that signal.

In [4]:
def weekend_features(df, prefix):
    df = df.copy()
    df['is_weekend'] = df['Date'].dt.dayofweek.isin([4, 5, 6]).astype('int8')
    df['is_booked'] = (df['Status'] == 'B').astype('int8')

    we = df[df['is_weekend'] == 1].groupby('PropertyID').agg(
        we_days=('Date', 'count'),
        we_booked=('is_booked', 'sum'),
        we_price_mean=('Price', 'mean'),
    )
    wd = df[df['is_weekend'] == 0].groupby('PropertyID').agg(
        wd_days=('Date', 'count'),
        wd_booked=('is_booked', 'sum'),
        wd_price_mean=('Price', 'mean'),
    )
    out = we.join(wd, how='outer').reset_index()
    out['we_booking_rate'] = out['we_booked'] / out['we_days']
    out['wd_booking_rate'] = out['wd_booked'] / out['wd_days']
    out['weekend_premium'] = out['we_price_mean'] / out['wd_price_mean'].replace(0, np.nan)
    out['weekend_minus_weekday_book_rate'] = out['we_booking_rate'] - out['wd_booking_rate']
    out.columns = ['PropertyID'] + [f'{prefix}_{c}' for c in out.columns[1:]]
    return out

we_q1 = weekend_features(listing_q1, 'q1')
we_q2 = weekend_features(listing_q2, 'q2')
print('we_q1:', we_q1.shape)
we_q1.head()

we_q1: (48690, 11)


,PropertyID,q1_we_days,q1_we_booked,q1_we_price_mean,q1_wd_days,q1_wd_booked,q1_wd_price_mean,q1_we_booking_rate,q1_wd_booking_rate,q1_weekend_premium,q1_weekend_minus_weekday_book_rate
0,105,39,0,782.333313,52,0,549.000000,0.000000,0.000000,1.425015,0.000000
1,795,24,0,300.000000,36,0,300.000000,0.000000,0.000000,1.000000,0.000000
2,2515,39,8,69.000000,52,14,66.692307,0.205128,0.269231,1.034602,-0.064103
3,2534,39,0,160.897430,52,0,156.057693,0.000000,0.000000,1.031013,0.000000
4,2539,39,0,67.717949,52,0,45.923077,0.000000,0.000000,1.474595,0.000000


### 2.3 Monthly aggregates
Track how each property's booking volume trended month-to-month, so the model can pick up acceleration/deceleration leading into Q3.

In [5]:
def monthly_booked_days(df, prefix):
    df = df.copy()
    df['month'] = df['Date'].dt.month
    df['is_booked'] = (df['Status'] == 'B').astype('int8')
    grouped = (df.groupby(['PropertyID', 'month'])['is_booked']
                 .sum().unstack(fill_value=0))
    grouped.columns = [f'{prefix}_booked_m{m}' for m in grouped.columns]
    return grouped.reset_index()

month_q1 = monthly_booked_days(listing_q1, 'q1')   # months 1-3
month_q2 = monthly_booked_days(listing_q2, 'q2')   # months 4-6
print('month_q1:', month_q1.shape, '| month_q2:', month_q2.shape)
month_q2.head()

month_q1: (48690, 4) | month_q2: (48690, 4)


,PropertyID,q2_booked_m4,q2_booked_m5,q2_booked_m6
0,105,0,0,0
1,795,0,0,0
2,2515,0,0,2
3,2534,1,9,30
4,2539,0,0,0


### 2.4 Status-transition counts
How often a property flips between Available / Booked / Reserved measures activity churn — a busy listing vs. a static one.

In [6]:
def status_transitions(df, prefix):
    df = df.sort_values(['PropertyID', 'Date']).copy()
    df['Status'] = df['Status'].astype(str)
    df['prev_status'] = df.groupby('PropertyID')['Status'].shift(1)
    df['changed'] = (df['Status'] != df['prev_status']) & df['prev_status'].notna()
    df['A_to_B'] = ((df['prev_status'] == 'A') & (df['Status'] == 'B')).astype('int8')
    df['B_to_A'] = ((df['prev_status'] == 'B') & (df['Status'] == 'A')).astype('int8')

    out = df.groupby('PropertyID').agg(
        status_changes=('changed', 'sum'),
        new_bookings=('A_to_B', 'sum'),
        booking_ends=('B_to_A', 'sum'),
    ).reset_index()
    out.columns = ['PropertyID'] + [f'{prefix}_{c}' for c in out.columns[1:]]
    return out

trans_q1 = status_transitions(listing_q1, 'q1')
trans_q2 = status_transitions(listing_q2, 'q2')
print('trans_q1:', trans_q1.shape)
trans_q2.head()

trans_q1: (48690, 4)


,PropertyID,q2_status_changes,q2_new_bookings,q2_booking_ends
0,105,0,0,0
1,795,0,0,0
2,2515,18,1,0
3,2534,7,1,1
4,2539,0,0,0


In [7]:
# 2.6 Demand Proxy (Q2 vs Q1)
# Calculating relative change of mean price between Q1 and Q2

def compute_demand_features(q1_df, q2_df):
    temp = q2_df[['PropertyID', 'q2_price_mean']].merge(
        q1_df[['PropertyID', 'q1_price_mean']], on='PropertyID', how='inner'
    )
    
    # Calculate relative change
    temp['price_change_q2_q1'] = (temp['q2_price_mean'] - temp['q1_price_mean']) / temp['q1_price_mean'].replace(0, np.nan)
    
    # Also add change in booking rates as a demand proxy
    temp_rate = q2_df[['PropertyID', 'q2_booking_rate']].merge(
        q1_df[['PropertyID', 'q1_booking_rate']], on='PropertyID', how='inner'
    )
    temp['booking_rate_change_q2_q1'] = temp_rate['q2_booking_rate'] - temp_rate['q1_booking_rate']
    
    return temp[['PropertyID', 'price_change_q2_q1', 'booking_rate_change_q2_q1']]

demand_features = compute_demand_features(base_q1, base_q2)
print('demand_features:', demand_features.shape)
demand_features.head()

demand_features: (48690, 3)


,PropertyID,price_change_q2_q1,booking_rate_change_q2_q1
0,105,0.000000,0.00000
1,795,0.000000,0.00000
2,2515,0.462737,-0.21978
3,2534,-0.069145,0.43956
4,2539,0.171008,0.00000


### 2.5 Recency window — last 30 days of Q2
Behaviour in late June is the most recent signal we have before Q3 starts.

In [8]:
cutoff = pd.Timestamp('2016-06-01')
recent = listing_q2[listing_q2['Date'] >= cutoff].copy()
recent['is_booked'] = (recent['Status'] == 'B').astype('int8')
recent['is_available'] = (recent['Status'] == 'A').astype('int8')

recent_agg = recent.groupby('PropertyID').agg(
    recent_days=('Date', 'count'),
    recent_booked=('is_booked', 'sum'),
    recent_available=('is_available', 'sum'),
    recent_price_mean=('Price', 'mean'),
    recent_price_std=('Price', 'std'),
).reset_index()
recent_agg['recent_booking_rate']   = recent_agg['recent_booked']   / recent_agg['recent_days']
recent_agg['recent_available_rate'] = recent_agg['recent_available'] / recent_agg['recent_days']
recent_agg.columns = ['PropertyID'] + [f'recent_{c}' if not c.startswith('recent_') else c
                                       for c in recent_agg.columns[1:]]
print('recent_agg:', recent_agg.shape)
recent_agg.head()

recent_agg: (48288, 8)


,PropertyID,recent_days,recent_booked,recent_available,recent_price_mean,recent_price_std,recent_booking_rate,recent_available_rate
0,105,30,0,30,642.333313,157.421753,0.000000,1.0
1,795,30,0,30,300.000000,0.000000,0.000000,1.0
2,2515,30,2,9,99.000000,0.000000,0.066667,0.3
3,2534,30,30,0,145.000000,0.000000,1.000000,0.0
4,2539,30,0,30,64.333336,8.995529,0.000000,1.0


### 2.6 Combined H1 aggregates (Q1 + Q2 together)
Some signals are best measured over the whole half-year window.

In [9]:
listing_h1 = pd.concat([listing_q1, listing_q2], ignore_index=True)
base_h1  = base_aggregates(listing_h1, 'h1')
trans_h1 = status_transitions(listing_h1, 'h1')
print('base_h1:', base_h1.shape, '| trans_h1:', trans_h1.shape)
del listing_h1

base_h1: (48690, 20) | trans_h1: (48690, 4)


## 3. Property-Metadata Feature Factory

In [10]:
def property_features(df):
    p = df.copy()

    # PropertyAge in days at the start of Q3
    q3_start = pd.Timestamp('2016-07-01')
    p['PropertyAgeDays'] = (q3_start - p['CreatedDate']).dt.days
    p['PropertyAgeDays'] = p['PropertyAgeDays'].clip(lower=0)

    # Reviews per month since creation (proxy for popularity)
    months_active = (p['PropertyAgeDays'] / 30.0).replace(0, np.nan)
    p['ReviewsPerMonth'] = p['NumberofReviews'] / months_active

    # Boolean encodings
    for col in ['Superhost', 'BusinessReady']:
        if col in p.columns:
            p[col] = p[col].map({True: 1, False: 0, 'TRUE': 1, 'FALSE': 0}).astype('float32')
    if 'InstantbookEnabled' in p.columns:
        p['InstantbookEnabled'] = (p['InstantbookEnabled'].astype(str).str.lower()
                                   .map({'yes': 1, 'no': 0}).astype('float32'))

    # Price-per-guest, price ratios
    if 'PublishedNightlyRate' in p.columns and 'MaxGuests' in p.columns:
        p['PricePerGuest'] = p['PublishedNightlyRate'] / p['MaxGuests'].replace(0, np.nan)
    if 'PublishedNightlyRate' in p.columns and 'PublishedWeeklyRate' in p.columns:
        p['WeeklyDiscount'] = 1 - (p['PublishedWeeklyRate'] / 7) / p['PublishedNightlyRate'].replace(0, np.nan)
    if 'PublishedNightlyRate' in p.columns and 'PublishedMonthlyRate' in p.columns:
        p['MonthlyDiscount'] = 1 - (p['PublishedMonthlyRate'] / 30) / p['PublishedNightlyRate'].replace(0, np.nan)

    # Bedrooms / bathrooms ratio
    if 'Bedrooms' in p.columns and 'Bathrooms' in p.columns:
        p['BedBathRatio'] = p['Bedrooms'] / p['Bathrooms'].replace(0, np.nan)

    # Missing indicators for fee/response columns
    for col in ['ResponseRate', 'ResponseTimemin', 'SecurityDeposit', 'CleaningFee',
                'ExtraPeopleFee', 'OverallRating']:
        if col in p.columns:
            p[f'{col}_missing'] = p[col].isna().astype('int8')

    # Has-reviews flag
    p['HasReviews'] = (p['NumberofReviews'] > 0).astype('int8')

    #  --- Text Keywords ---
    if 'ListingTitle' in p.columns:
        title = p['ListingTitle'].fillna('').str.lower()
        p['kw_luxury']  = title.str.contains('luxury|upscale|fancy|premium').astype('int8')
        p['kw_private'] = title.str.contains('private|quiet|secluded|separate').astype('int8')
        p['kw_near']    = title.str.contains('near|close to|steps to|minutes to').astype('int8')
        p['kw_cozy']    = title.str.contains('cozy|charming|warm|homey').astype('int8')
        p['kw_view']    = title.str.contains('view|scenic|panorama').astype('int8')
        p['kw_modern']  = title.str.contains('modern|stylish|renovated|new').astype('int8')
        p['title_len']  = title.str.len()
        p['title_words'] = title.str.split().str.len()

    # --- Location Clustering  ---
    if 'Latitude' in p.columns and 'Longitude' in p.columns:
        coords = p[['Latitude', 'Longitude']].fillna(p[['Latitude', 'Longitude']].median())
        kmeans = KMeans(n_clusters=12, random_state=RANDOM_STATE, n_init=10)
        p['geo_cluster'] = kmeans.fit_predict(coords).astype(str) # category-like

    drop_cols = ['HostID', 'ListingTitle', 'Country', 'State', 'City', 'CreatedDate']
    p = p.drop(columns=[c for c in drop_cols if c in p.columns])
    return p

prop_feat = property_features(property_info)
print('prop_feat:', prop_feat.shape)
prop_feat.head()

prop_feat: (48690, 49)


,PropertyID,PropertyType,ListingType,Zipcode,Neighborhood,MetropolitanStatisticalArea,NumberofReviews,OverallRating,Bedrooms,Bathrooms,MaxGuests,ResponseRate,ResponseTimemin,Superhost,CancellationPolicy,SecurityDeposit,CleaningFee,ExtraPeopleFee,PublishedNightlyRate,PublishedMonthlyRate,PublishedWeeklyRate,MinimumStay,NumberofPhotos,BusinessReady,InstantbookEnabled,Latitude,Longitude,PropertyAgeDays,ReviewsPerMonth,PricePerGuest,WeeklyDiscount,MonthlyDiscount,BedBathRatio,ResponseRate_missing,ResponseTimemin_missing,SecurityDeposit_missing,CleaningFee_missing,ExtraPeopleFee_missing,OverallRating_missing,HasReviews,kw_luxury,kw_private,kw_near,kw_cozy,kw_view,kw_modern,title_len,title_words,geo_cluster
0,7621748,Apartment,Private room,10001,Chelsea,"New York-Newark-Jersey City, NY-NJ-PA Metro Area",27.0,4.5,1.0,2.0,2,NaN,NaN,NaN,Strict,100.0,40.0,25.0,104,2912.0,728.0,1,4.0,1.0,1.0,40.747016,-73.993048,340,2.382353,52.000000,0.000000,0.066667,0.5,1,1,0,0,0,0,1,0,0,0,0,0,0,34,5,8
1,4205815,Apartment,Private room,10001,Chelsea,"New York-Newark-Jersey City, NY-NJ-PA Metro Area",103.0,4.0,1.0,1.0,2,100.0,0.03,0.0,Strict,NaN,NaN,5.0,110,3080.0,770.0,1,11.0,0.0,1.0,40.752395,-74.002176,655,4.717557,55.000000,0.000000,0.066667,1.0,0,0,1,1,0,0,1,0,0,0,0,0,0,33,8,8
2,7661915,Apartment,Private room,10011,Chelsea,"New York-Newark-Jersey City, NY-NJ-PA Metro Area",41.0,4.9,1.0,1.0,2,100.0,18.72,0.0,Moderate,200.0,35.0,NaN,210,4900.0,1200.0,2,14.0,0.0,0.0,40.745535,-73.997054,338,3.639053,105.000000,0.183673,0.222222,1.0,0,0,0,0,1,0,1,1,0,0,0,0,0,33,5,8
3,300140,Apartment,Entire home/apt,10001,Chelsea,"New York-Newark-Jersey City, NY-NJ-PA Metro Area",170.0,4.6,2.0,1.0,4,100.0,51.28,0.0,Moderate,NaN,NaN,NaN,310,6700.0,2170.0,2,16.0,1.0,0.0,40.753051,-73.999257,1643,3.104078,77.500000,0.000000,0.279570,2.0,0,0,1,1,1,0,1,0,0,0,1,0,0,29,6,8
4,4129195,Apartment,Entire home/apt,10199,Chelsea,"New York-Newark-Jersey City, NY-NJ-PA Metro Area",61.0,4.8,2.0,1.0,6,100.0,93.40,0.0,Strict,200.0,250.0,NaN,475,4950.0,1575.0,5,11.0,1.0,0.0,40.750971,-73.995618,661,2.768533,79.166667,0.526316,0.652632,2.0,0,0,0,0,1,0,1,0,0,0,0,0,0,34,9,8


## 4. Assemble Final Train / Test Feature Matrices

In [11]:
feature_tables = [prop_feat,
                  base_q1, base_q2, base_h1,
                  we_q1,   we_q2,
                  month_q1, month_q2,
                  trans_q1, trans_q2, trans_h1,
                  recent_agg,
                  demand_features] # Added demand_features to the list of feature tables

def assemble(id_df):
    out = id_df.copy()
    for t in feature_tables:
        out = out.merge(t, on='PropertyID', how='left')
    return out

train_df = assemble(train_y)
test_df  = assemble(test_ids)
print('train_df:', train_df.shape, '| test_df:', test_df.shape)
train_df.head()

train_df: (24372, 151) | test_df: (24318, 150)


,PropertyID,NumReserveDays2016Q3,PropertyType,ListingType,Zipcode,Neighborhood,MetropolitanStatisticalArea,NumberofReviews,OverallRating,Bedrooms,Bathrooms,MaxGuests,ResponseRate,ResponseTimemin,Superhost,CancellationPolicy,SecurityDeposit,CleaningFee,ExtraPeopleFee,PublishedNightlyRate,PublishedMonthlyRate,PublishedWeeklyRate,MinimumStay,NumberofPhotos,BusinessReady,InstantbookEnabled,Latitude,Longitude,PropertyAgeDays,ReviewsPerMonth,PricePerGuest,WeeklyDiscount,MonthlyDiscount,BedBathRatio,ResponseRate_missing,ResponseTimemin_missing,SecurityDeposit_missing,CleaningFee_missing,ExtraPeopleFee_missing,OverallRating_missing,HasReviews,kw_luxury,kw_private,kw_near,kw_cozy,kw_view,kw_modern,title_len,title_words,geo_cluster,...,h1_booking_rate,h1_available_rate,h1_reserved_rate,h1_price_range,h1_price_cv,h1_price_iqr,q1_we_days,q1_we_booked,q1_we_price_mean,q1_wd_days,q1_wd_booked,q1_wd_price_mean,q1_we_booking_rate,q1_wd_booking_rate,q1_weekend_premium,q1_weekend_minus_weekday_book_rate,q2_we_days,q2_we_booked,q2_we_price_mean,q2_wd_days,q2_wd_booked,q2_wd_price_mean,q2_we_booking_rate,q2_wd_booking_rate,q2_weekend_premium,q2_weekend_minus_weekday_book_rate,q1_booked_m1,q1_booked_m2,q1_booked_m3,q2_booked_m4,q2_booked_m5,q2_booked_m6,q1_status_changes,q1_new_bookings,q1_booking_ends,q2_status_changes,q2_new_bookings,q2_booking_ends,h1_status_changes,h1_new_bookings,h1_booking_ends,recent_days,recent_booked,recent_available,recent_price_mean,recent_price_std,recent_booking_rate,recent_available_rate,price_change_q2_q1,booking_rate_change_q2_q1
0,105,0,Apartment,Private room,10036,Hell's Kitchen,"New York-Newark-Jersey City, NY-NJ-PA Metro Area",39.0,4.8,1.0,1.0,7,100.0,14.55,0.0,Flexible,NaN,NaN,150.0,549,15000.0,5000.0,1,25.0,0.0,0.0,40.762099,-73.996022,2925,0.400000,78.428571,-0.301067,0.089253,1.0,0,0,1,1,0,0,1,0,0,0,0,0,0,24,4,8,...,0.000000,1.000000,0.000000,350.0,0.244299,350.0,39,0,782.333313,52,0,549.000000,0.000000,0.000000,1.425015,0.000000,39,0,782.333313,52,0,549.000000,0.000000,0.000000,1.425015,0.000000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,30.0,0.0,30.0,642.333313,157.421753,0.0,1.000000,0.000000,0.000000
1,2534,0,Apartment,Entire home/apt,10065,Upper East Side,"New York-Newark-Jersey City, NY-NJ-PA Metro Area",1.0,3.0,1.0,1.0,2,NaN,NaN,0.0,Moderate,500.0,25.0,NaN,145,4060.0,1015.0,3,16.0,0.0,0.0,40.762059,-73.961365,2855,0.010508,72.500000,0.000000,0.066667,1.0,1,1,0,0,1,0,1,0,1,0,1,0,0,35,6,8,...,0.219780,0.043956,0.736264,115.0,0.096744,20.0,39,0,160.897430,52,0,156.057693,0.000000,0.000000,1.031013,0.000000,39,16,147.051285,52,24,147.307693,0.410256,0.461538,0.998259,-0.051282,0,0,0,1,9,30,0,0,0,7,1,1,7,1,1,30.0,30.0,0.0,145.000000,0.000000,1.0,0.000000,-0.069145,0.439560
2,2539,35,Apartment,Private room,11218,Kensington,"New York-Newark-Jersey City, NY-NJ-PA Metro Area",3.0,5.0,1.0,1.0,5,100.0,1.98,0.0,Moderate,250.0,25.0,25.0,64,999.0,299.0,1,12.0,0.0,0.0,40.647486,-73.972370,2854,0.031535,12.800000,0.332589,0.479688,1.0,0,0,0,0,0,0,1,0,1,0,0,0,0,34,8,11,...,0.000000,0.994505,0.005495,40.0,0.226519,30.0,39,0,67.717949,52,0,45.923077,0.000000,0.000000,1.474595,0.000000,39,0,72.333336,52,0,59.000000,0.000000,0.000000,1.225989,0.000000,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,30.0,0.0,30.0,64.333336,8.995529,0.0,1.000000,0.171008,0.000000
3,3330,43,Apartment,Private room,11206,Williamsburg,"New York-Newark-Jersey City, NY-NJ-PA Metro Area",21.0,4.8,1.0,1.0,2,95.0,550.12,0.0,Strict,150.0,75.0,50.0,175,1900.0,650.0,3,32.0,0.0,0.0,40.708558,-73.942362,2792,0.225645,87.500000,0.469388,0.638095,1.0,0,0,0,0,0,0,1,0,0,0,0,0,0,34,5,2,...,0.010989,0.159341,0.829670,25.0,0.063981,0.0,39,2,85.846153,52,0,85.384613,0.051282,0.000000,1.005405,0.051282,39,0,90.000000,52,0,90.192307,0.000000,0.000000,0.997868,0.000000,2,0,0,0,0,0,4,1,2,1,0,0,5,1,2,30.0,0.0,1.0,90.333336,1.825742,0.0,0.033333,0.052902,-0.021978
4,3831,76,Other,Entire home/apt,11238,Clinton Hill,"New York-Newark-Jersey City, NY-NJ-PA Metro Area",87.0,4.5,1.0,1.0,3,100.0,31.38,0.

In [12]:
# Quick column type inventory
n_num = train_df.select_dtypes(include='number').shape[1]
n_obj = train_df.select_dtypes(include='object').shape[1]
print(f'Numeric columns : {n_num}')
print(f'Object columns  : {n_obj}')
print('\nObject columns ->', train_df.select_dtypes(include='object').columns.tolist())

Numeric columns : 145
Object columns  : 6

Object columns -> ['PropertyType', 'ListingType', 'Neighborhood', 'MetropolitanStatisticalArea', 'CancellationPolicy', 'geo_cluster']


In [13]:
# Verify if keyword features are present
kw_cols = [c for c in train_df.columns if c.startswith('kw_') or c in ['title_len', 'word_count']]
print(f"Text-based features found: {kw_cols}")
train_df[kw_cols].head()

Text-based features found: ['kw_luxury', 'kw_private', 'kw_near', 'kw_cozy', 'kw_view', 'kw_modern', 'title_len']


,kw_luxury,kw_private,kw_near,kw_cozy,kw_view,kw_modern,title_len
0,0,0,0,0,0,0,24
1,0,1,0,1,0,0,35
2,0,1,0,0,0,0,34
3,0,0,0,0,0,0,34
4,0,0,0,1,0,0,31


## 5. Sanity Checks

In [14]:
# 5.1 Row counts must match input
assert len(train_df) == len(train_y), 'train row count mismatch'
assert len(test_df)  == len(test_ids), 'test row count mismatch'
print('Row counts OK.')

# 5.2 No leakage of target into test
assert 'NumReserveDays2016Q3' not in test_df.columns
print('No target leak in test.')

# 5.3 Train/test PropertyID disjoint
overlap = set(train_df['PropertyID']) & set(test_df['PropertyID'])
assert len(overlap) == 0, f'{len(overlap)} overlapping IDs!'
print('Train/test IDs disjoint.')

Row counts OK.
No target leak in test.
Train/test IDs disjoint.


In [15]:
# 5.4 Missing-value report (top 20)
mv = train_df.isna().mean().sort_values(ascending=False)
mv = mv[mv > 0].head(20)
print('Top columns by missing-rate in train:')
print((mv * 100).round(2).astype(str) + ' %')

Top columns by missing-rate in train:
ExtraPeopleFee           59.38 %
SecurityDeposit          52.93 %
ResponseRate             40.06 %
ResponseTimemin          37.53 %
CleaningFee              29.27 %
OverallRating            24.76 %
Superhost                15.33 %
WeeklyDiscount            1.15 %
MonthlyDiscount           1.15 %
PublishedWeeklyRate       1.15 %
PublishedMonthlyRate      1.15 %
BedBathRatio              1.14 %
recent_days               0.78 %
recent_booked             0.78 %
recent_available          0.78 %
recent_price_mean         0.78 %
recent_price_std          0.78 %
recent_booking_rate       0.78 %
recent_available_rate     0.78 %
Bathrooms                 0.56 %
dtype: str


In [16]:
# 5.5 Correlation of engineered features with target — preview top 25
num_train = train_df.select_dtypes(include='number').drop(columns=['PropertyID'])
corr = num_train.corr(numeric_only=True)['NumReserveDays2016Q3'].drop('NumReserveDays2016Q3')
corr = corr.reindex(corr.abs().sort_values(ascending=False).index)
print('Top-25 features by |correlation| with target:')
corr.head(25)

Top-25 features by |correlation| with target:


q2_days_R                  0.778497
q2_reserved_rate           0.777015
h1_reserved_rate           0.750516
q2_unique_reservations     0.745815
h1_days_R                  0.745723
h1_unique_reservations     0.720650
ReviewsPerMonth            0.703267
NumberofReviews            0.620932
h1_status_changes          0.586507
q1_unique_reservations     0.565608
q2_status_changes          0.561178
q1_days_R                  0.548181
q1_reserved_rate           0.548147
q1_status_changes          0.471582
ResponseRate_missing      -0.461255
ResponseTimemin_missing   -0.438003
h1_price_nunique           0.408836
q2_price_nunique           0.403285
q1_price_nunique           0.374238
q2_we_booking_rate        -0.353562
q2_we_booked              -0.352711
q2_booking_rate           -0.352695
q2_days_B                 -0.351794
recent_booked             -0.351522
recent_booking_rate       -0.351522
Name: NumReserveDays2016Q3, dtype: float64

## 6. Persist to Disk

In [17]:
from sklearn.model_selection import train_test_split

# Section IV.C: 80/20 Local Split (Local Train and Local Test)
# We split the labeled training data once here to ensure all models use the exact same
# training set (for 5-fold CV) and local test set (for final unbiased evaluation).
train_local, test_local = train_test_split(
    train_df, 
    test_size=0.20, 
    random_state=RANDOM_STATE,
    shuffle=True
)

print(f'Split complete:')
print(f'  Local Train set (80%): {train_local.shape}')
print(f'  Local Test set  (20%): {test_local.shape}')


Split complete:
  Local Train set (80%): (19497, 151)
  Local Test set  (20%): (4875, 151)


In [18]:
# Persist using parquet (compact + dtype-preserving). Fallback to csv if pyarrow missing.
try:
    # Full files
    train_df.to_parquet(OUT_DIR / 'train_features.parquet', index=False)
    test_df .to_parquet(OUT_DIR / 'test_features.parquet',  index=False)
    
    # Section IV.C split files (Local Train and Local Test)
    train_local.to_parquet(OUT_DIR / 'train_local.parquet', index=False)
    test_local .to_parquet(OUT_DIR / 'test_local.parquet', index=False)
    
    print('Saved all parquet files (full + split) to', OUT_DIR.resolve())
except Exception as e:
    print('Parquet failed, falling back to CSV:', e)
    train_df.to_csv(OUT_DIR / 'train_features.csv', index=False)
    test_df .to_csv(OUT_DIR / 'test_features.csv',  index=False)
    train_local.to_csv(OUT_DIR / 'all_train_local.csv', index=False)
    test_local .to_csv(OUT_DIR / 'all_test_local.csv', index=False)
    print('Saved CSV files to', OUT_DIR.resolve())


Saved all parquet files (full + split) to /Users/bashkal/Desktop/Comp468-ML_in_Python/ali/ML-Final/outputs


In [19]:
# 6. Feature Summary
print(f"Final Train Shape: {train_df.shape}")
print(f"Final Test Shape:  {test_df.shape}")

n_numeric = train_df.select_dtypes(include='number').shape[1]
n_object = train_df.select_dtypes(exclude='number').shape[1]

print(f"Numeric features:  {n_numeric}")
print(f"Categorical/Other: {n_object}")
print(f"Target variable included: {'NumReserveDays2016Q3' in train_df.columns}")


Final Train Shape: (24372, 151)
Final Test Shape:  (24318, 150)
Numeric features:  145
Categorical/Other: 6
Target variable included: True


## Summary

| Feature group | # of features (approx.) | Source | Status |
|---|---|---|---|
| Property metadata | ~30 | `property_info.csv` | Done |
| Text Keywords (Luxury, etc) | 8 | `ListingTitle` | Done |
| Base aggregates Q1 / Q2 / H1 | 3 × 16 ≈ 48 | listing tables | Done |
| Weekend / weekday split Q1 / Q2 | 2 × 8 = 16 | listing tables | Done |
| Monthly booked days | 6 | listing tables | Done |
| Status transitions Q1 / Q2 / H1 | 9 | listing tables | Done |
| Recency (last 30 days of Q2) | 7 | listing_q2 | Done |
| Demand Proxy (Q1 vs Q2 change) | 2 | listing_q1 + listing_q2 | Done |

Roughly **118–120 numeric features** + a handful of categoricals (PropertyType, ListingType, Neighborhood, Zipcode, CancellationPolicy) which the modelling notebooks will OneHot- or target-encode inside the pipeline.

### Next Step → `03_Baseline.ipynb`
Build the modelling pipeline:
1. `ColumnTransformer` (median impute numerics + missing flag, OneHot encode low-cardinality cats, target encode high-cardinality cats).
2. Train **Linear Regression → RandomForest → GradientBoosting** with 5-fold CV.
3. Report MSE per model, save out-of-fold predictions for stacking later.
4. Produce a first Kaggle submission file from the best baseline.